In [42]:
import pandas as pd
import numpy as np
import re, unicodedata

from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV

from sentence_transformers import SentenceTransformer

import xgboost as xgb
from tqdm.auto import tqdm

import joblib
import os

## Prediccion de precio con valor 449 euros

In [61]:
# Crear un nuevo dataframe con todas las columnas especificadas
new_product = pd.DataFrame({
    'product_title': ['Dell 15 Ordenador Portátil DC15255 15.6" FHD (1920 x 1080) 120Hz, Procesador AMD Ryzen 5 7520U, Gráficos Radeon 610M, 8GB LPDDR5 RAM, 512GB SSD, Windows 11 Home, Teclado QWERTY Espanol - Negro'],
    'product_rating': [4.4],
    'is_best_seller': ["Best Seller"],
    'is_sponsored': ['Organic'],
    'buy_box_availability': [1],
    'sustainability_tags': [1],
    'has_coupon': [0],
    'discount_percentage': [0.0],
    'product_category': ['Laptops'],
    'product_segment': ['Media'],
    'log_purchased_last_month': [np.log1p(50)],
    'log_total_reviews': [np.log1p(113)]
})

new_product

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_purchased_last_month,log_total_reviews
0,"Dell 15 Ordenador Portátil DC15255 15.6"" FHD (...",4.4,Best Seller,Organic,1,1,0,0.0,Laptops,Media,3.931826,4.736198


## Predicción para un producto de precio 15 euros

In [65]:
# Crear un nuevo dataframe con todas las columnas especificadas
new_product = pd.DataFrame({
    'product_title': ['Stouchi 10K 8K Cable HDMI 2.1 de 2M,Certificado 48Gbps de Velocidad Ultra Alta 8K@60 4K@120/144Hz RTX 3080 eARC HDR10 HDCP2.2&2.3 Dolby Compatible con PS 5/Xbox Series X/Samsung/Sony/LG'],
    'product_rating': [4.5],
    'is_best_seller': ["Amazon's"],
    'is_sponsored': ['Sponsored'],
    'buy_box_availability': [1],
    'sustainability_tags': [1],
    'has_coupon': [0],
    'discount_percentage': [0.0],
    'product_category': ['Chargers, Adapters & Cables'],
    'product_segment': ['Baja'],
    'log_purchased_last_month': [np.log1p(0)],
    'log_total_reviews': [np.log1p(19792)]
})

new_product

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_purchased_last_month,log_total_reviews
0,"Stouchi 10K 8K Cable HDMI 2.1 de 2M,Certificad...",4.5,Amazon's,Sponsored,1,1,0,0.0,"Chargers, Adapters & Cables",Baja,0.0,9.893084


In [66]:
# 1. Definimos una lista de "Palabras que parecen marcas pero son ruido"
blacklist = ['EZ', 'THE', 'AND', 'FOR', 'PRO', 'NEW', 'OFF']

# 2. Tu proceso de extracción
new_product['brand'] = new_product['product_title'].str.split().str[0].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)

# 3. FILTRO DE CALIDAD:
# Enviamos a OTHER si:
# - Está en la blacklist
# - Es un número puro (ej: "100")
# - Es demasiado corto (menos de 2 caracteres, excepto marcas como LG)
def clean_brand(b):
    if b in blacklist: return 'OTHER'
    if b.isdigit(): return 'OTHER' 
    if len(b) < 2: return 'OTHER'
    return b

new_product['brand'] = new_product['brand'].apply(clean_brand)

# 4. RE-CALCULAR EL TOP 50
# Tip: Si quieres que GIGABYTE sea 'OTHER' podrías sacarla del conteo manualmente
top_brands = new_product[new_product['brand'] != 'OTHER']['brand'].value_counts().nlargest(100).index
new_product['brand_cat'] = new_product['brand'].apply(lambda x: x if x in top_brands else 'OTHER')
new_product.drop(columns=['brand'], inplace=True)

# 2. BOOLEAN FEATURES (Regex)
# si es un modelo de alta gama
new_product['is_high_end'] = new_product['product_title'].str.contains(r'\b(Pro|Ultra|Gaming|Business|Elite|Max|Plus|Enterprise)\b', case=False, regex=True).astype(int)

def extraer_resolucion(title):
    title = str(title).lower()
    
    # 1. NIVEL TOP (8K, 5K - Muy caros)
    if re.search(r'\b(8k|5k)\b', title):
        return '8K_5K_Ultra'
        
    # 2. NIVEL ALTO (4K, UHD, Retina - Apple y Gama Alta)
    # "Retina" es clave para Apple, que no suele decir "4K" pero es caro.
    if re.search(r'\b(4k|uhd|2160p|ultra hd|retina)\b', title):
        return '4K_UHD_Retina'
        
    # 3. NIVEL MEDIO-ALTO (2K, QHD, 1440p - Gaming y Oficina Pro)
    if re.search(r'\b(2k|qhd|wqhd|1440p)\b', title):
        return '2K_QHD'
        
    # 4. NIVEL ESTÁNDAR (Full HD, 1080p - El estándar hoy en día)
    if re.search(r'\b(1080p|fhd|full hd|1920x1080)\b', title):
        return 'FHD_1080p'
        
    # 5. NIVEL BÁSICO (HD, 720p - Laptops baratos y TVs pequeñas)
    if re.search(r'\b(720p|hd|1366x768)\b', title):
        return 'HD_Basic'
        
    # Si no especifica nada (NaN para que XGBoost decida)
    return np.nan


def extractor_maestro(df):
    # Creamos copias para no tocar el original
    df_new = df.copy()
    
    # 1. Definimos las nuevas columnas vacías (NaN)
    specs = ['spec_ram_gb', 'spec_storage_gb', 'spec_screen_inch', 
    'spec_power_w', 'spec_pack_count', 'spec_resolution_cat',
    'spec_refresh_rate_hz', 'spec_ram_speed_mhz']
    for col in specs:
        df_new[col] = np.nan

    # 2. Lógica de Extracción Fila a Fila
    for i, row in df_new.iterrows():
        title = str(row['product_title']).lower()
        cat = str(row['product_category'])
        
        # --- BLOQUE A: RAM (Solo para Ordenadores/Tablets) ---
        if cat in ['Laptops', 'PC Components', 'Tablets & E-readers (DEVICES ONLY)', 'Computer Peripherals']:
            
            # EXPLICACIÓN REGEX RAM:
            # 1. (\d+)\s*(?:gb|g) -> Coge numero y unidad
            # 2. Grupo de Ruido con LOOKAHEAD NEGATIVO (?!) -> Acepta palabras intermedias SOLO SI:
            #    a) NO son 'ssd', 'hdd', 'storage', 'rom' (Evita coger discos)
            #    b) NO son 'd+ gb' (Evita coger otra capacidad que aparezca después)
            
            regex_ram = r'(\d+)\s*(?:gb|g)\s*(?:(?!(?:ssd|hdd|storage|rom)|\d+\s*(?:gb|g))[\w\-\.\(\)]+\s*){0,3}?(?:ram|memory|ddr|unified|vram|gddr)|(?:ram|memory|ddr\d?|capacity|vram|gddr\d?)\s*(?:kit)?\s*(\d+)\s*(?:gb|g)'
            
            match = re.search(regex_ram, title)
            if match:
                val = next((m for m in match.groups() if m is not None), None)
                if val: df_new.at[i, 'spec_ram_gb'] = float(val)
                
        # --- BLOQUE B: ALMACENAMIENTO (Ordenadores, Discos, Móviles) ---
        target_cats_storage = ['Laptops', 'PC Components', 'Storage & Memory Cards', 'Mobile Cell Phones & Smartphones', 'Tablets & E-readers (DEVICES ONLY)']
        if cat in target_cats_storage:
            
            # CAMBIO CLAVE EN EL REGEX:
            # Antes: (?!gb|tb|to) -> Solo miraba si empezaba por letras de unidad.
            # Ahora: (?!\d+\s*(?:gb|tb|to)) -> Mira si es un NÚMERO seguido de unidad.
            # Esto impide que "512GB" sea tratado como ruido.
            
            regex_storage = r'(\d+)\s*(gb|tb|to)\s*(?:(?!\d+\s*(?:gb|tb|to))[\w\-\.]+\s*){0,3}?(?:ssd|hdd|storage|flash|rom|emmc|hard drive|disk)'
            
            match = re.search(regex_storage, title)
            
            # Fallback para tarjetas de memoria
            if not match and cat == 'Storage & Memory Cards':
                match = re.search(r'(\d+)\s*(gb|tb|to)', title)
            
            if match:
                val = float(match.group(1))
                unit = match.group(2)
                
                if unit in ['tb', 'to']:
                    val *= 1024
                
                df_new.at[i, 'spec_storage_gb'] = val

        # C. PANTALLA Y RESOLUCIÓN (Visuales)
        if cat in ['Laptops', 'TV & Video Displays', 'Monitors', 'Tablets & E-readers (DEVICES ONLY)', 'Mobile Cell Phones & Smartphones', 'Office Supplies, Ink & Toner']:
            # Pulgadas
            match = re.search(r'(\d+(?:\.\d+)?)\s*(?:\"|inch|”|\-inch)', title)
            if match: df_new.at[i, 'spec_screen_inch'] = float(match.group(1))
            
            # Resolución (Nueva Lógica)
            res = extraer_resolucion(title)
            if pd.notna(res): df_new.at[i, 'spec_resolution_cat'] = res

        # --- BLOQUE D: POTENCIA (Cargadores, Audio, Componentes) ---
        if cat in ['Chargers, Adapters & Cables', 'Power & Batteries', 'Audio, Sound & Recording Gear', 'PC Components', 'Computer Peripherals']:
            # \b significa "Word Boundary" (Límite de palabra). 
            # Esto evita que 'Wireless', 'White', 'Warranty' o 'With' cuenten como 'W'.
            
            # Busca: Numero + (W o Watt o Vatios) + FINAL DE PALABRA
            match = re.search(r'(\d+)\s*(?:w|watt|vatios)\b', title)
            
            if match:
                df_new.at[i, 'spec_power_w'] = float(match.group(1))

        # --- BLOQUE E: PACKS (Oficina, Cables, Pilas) ---
        if cat in ['Office Supplies, Ink & Toner', 'Small Gadget Accessories (Cases & Protectors)', 
                   'Chargers, Adapters & Cables', 'Power & Batteries', 'Cameras & Photography', 
                   'Computer Peripherals', 'Other Electronics']:
            
            # 1. Patrón Estándar: "Pack of 2", "Set of 5"
            match = re.search(r'(?:pack of|set of|count of)\s*(\d+)', title)
            
            # 2. Patrón Inverso: "2-Pack", "10 count", "50 sheets", "2 pieces", "4 cartridges"
            # AÑADIDO: 'cartridges' a la lista de palabras clave
            if not match: 
                match = re.search(r'(\d+)\s*[-]?\s*(?:pack|set|count|pcs|sheets|pieces|cartridges)', title)
            
            # 3. Patrón Tinta Entre Paréntesis (CASO ESPECÍFICO PGBK)
            # Detecta: "(2 PGBK)", "(4 BK)", "(2 Black)", "(5 Cartridges)"
            # La clave son los paréntesis \(\) para no confundir con modelos (ej: Canon 280)
            if not match and cat == 'Office Supplies, Ink & Toner':
                match = re.search(r'\(\s*(\d+)\s*(?:pgbk|bk|c|m|y|black|cyan|magenta|yellow|color|ink)\b', title)

            if match:
                df_new.at[i, 'spec_pack_count'] = float(match.group(1))
            
            # 4. Caso especial Pair/Twin
            elif 'pair' in title or 'twin pack' in title: 
                df_new.at[i, 'spec_pack_count'] = 2.0
                
        # Buscamos Hz altos (90, 120, 144, 165, 240, 360, etc.)
        if cat in ['Laptops', 'TV & Video Displays', 'Monitors', 'Tablets & E-readers (DEVICES ONLY)', 'Mobile Cell Phones & Smartphones']:
            # Regex: Número + Hz (con boundary \b para no coger cosas raras)
            match_hz = re.search(r'(\d+)\s*hz\b', title)
            if match_hz:
                hz_val = float(match_hz.group(1))
                # FILTRO DE SEGURIDAD:
                # Ignoramos 50Hz o 60Hz si es una TV barata o Laptop, ya que es el estándar y a veces se confunde con el input eléctrico.
                # Nos interesan los valores "Gaming" que suben el precio.
                # O bien, lo guardamos todo y dejamos que XGBoost decida. Yo recomiendo guardarlo todo.
                df_new.at[i, 'spec_refresh_rate_hz'] = hz_val

        # 2. VELOCIDAD RAM (PC Components, Laptops)
        # Buscamos MHz (2666, 3200, 5200, 6000...)
        if cat in ['PC Components', 'Laptops', 'Computer Peripherals']:
            
            # Regex: Número + (mhz O mt/s O mts)
            # (?i) al principio hace que sea case-insensitive (aunque ya pasamos title.lower())
            match_speed = re.search(r'(\d+)\s*(?:mhz|mt/s|mts)\b', title)
            
            if match_speed:
                val = float(match_speed.group(1))
                
                # FILTRO ANTI-RUIDO:
                # A veces coge "2.4 ghz" (wifi) como 2 (mhz). 
                # Las RAMs modernas suelen ser de más de 1000 MHz.
                # Las viejas DDR2 eran 400-800. 
                # Vamos a poner un suelo de 200 para evitar coger frecuencias de radio o wifi mal escritas.
                if val > 200:
                    df_new.at[i, 'spec_ram_speed_mhz'] = val
        

    return df_new


new_product = extractor_maestro(new_product)

C:\Users\diego\AppData\Local\Temp\ipykernel_18396\1013970887.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  new_product['is_high_end'] = new_product['product_title'].str.contains(r'\b(Pro|Ultra|Gaming|Business|Elite|Max|Plus|Enterprise)\b', case=False, regex=True).astype(int)


### TF-IDF

In [40]:
tfidf = joblib.load('Archivos_modelos/tfidf_vectorizer.pkl')
svd = joblib.load('Archivos_modelos/lsa_svd_model.pkl')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text

# Limpiar el título
clean_title = clean_text(new_product['product_title'].iloc[0])

# IMPORTANTE: Usamos .transform(), NO .fit_transform()
tfidf_vector = tfidf.transform([clean_title])
lsa_vector = svd.transform(tfidf_vector)

# Crear columnas LSA
n_components = 15
lsa_cols = [f'lsa_{i}' for i in range(n_components)]
df_lsa = pd.DataFrame(lsa_vector, columns=lsa_cols, index=new_product.index)

# Unir con el producto original
df_final = pd.concat([new_product, df_lsa], axis=1)

# 4. PREPARACIÓN FINAL PARA XGBOOST
cols_to_keep = [
    'product_title','product_rating', 'log_total_reviews', 'log_purchased_last_month', 
    'is_sponsored', 'buy_box_availability', 'has_coupon', 'discount_percentage', 
    'product_category', 'brand_cat', 'is_high_end', 'spec_ram_gb', 'spec_storage_gb',
    'spec_screen_inch', 'spec_power_w', 'spec_pack_count', 'spec_resolution_cat', 
    'spec_refresh_rate_hz', 'spec_ram_speed_mhz', 'is_best_seller', 'product_segment'
] + lsa_cols

# Seleccionar y ordenar columnas exactamente como las vio el modelo
new_product = df_final[cols_to_keep].copy()

# Convertir a categorías (XGBoost necesita esto si no usaste OHE)
cat_features = ['product_category', 'brand_cat', 'spec_resolution_cat']
for col in cat_features:
    new_product[col] = new_product[col].astype('category')

print("✅ Producto listo para predicción.")
print(f"Dimensiones: {new_product.shape}") # Debería ser (1, 36) aprox.

✅ Producto listo para predicción.
Dimensiones: (1, 36)


### Wjunwei

In [67]:
model_ecom = SentenceTransformer('wjunwei/ecommerce_text_embedding')
# Cargamos el PCA que guardaste en el entrenamiento
pca_emb = joblib.load('Archivos_modelos/pca_embeddings_model_50.pkl')

# 2. PROCESAMIENTO SEMÁNTICO (Embeddings + PCA)
# Nota: No usamos clean_text porque el Transformer aprovecha mayúsculas y puntuación
titulo_producto = str(new_product['product_title'].iloc[0])

# Generar el embedding (vector de alta dimensión)
# Usamos [titulo] para que sea una lista de un solo elemento
embedding_vector = model_ecom.encode([titulo_producto])

# Reducir a 50 componentes usando el PCA cargado
compressed_embedding = pca_emb.transform(embedding_vector)

# Crear columnas de Embeddings (emb_0, emb_1, ... emb_49)
n_components_emb = 50
emb_cols = [f'emb_{i}' for i in range(n_components_emb)]
df_emb = pd.DataFrame(compressed_embedding, columns=emb_cols, index=new_product.index)

# Unir con el producto original
df_final = pd.concat([new_product, df_emb], axis=1)

# 3. PREPARACIÓN FINAL PARA XGBOOST
# IMPORTANTE: Asegúrate de que esta lista coincida exactamente con la del entrenamiento
cols_to_keep = [
    'product_title','product_rating', 'log_total_reviews', 'log_purchased_last_month', 
    'is_sponsored', 'buy_box_availability', 'has_coupon', 'discount_percentage', 
    'product_category', 'brand_cat', 'is_high_end', 'spec_ram_gb', 'spec_storage_gb',
    'spec_screen_inch', 'spec_power_w', 'spec_pack_count', 'spec_resolution_cat', 
    'spec_refresh_rate_hz', 'spec_ram_speed_mhz', 'is_best_seller', 'product_segment'
] + emb_cols

# Seleccionar y ordenar columnas
new_product_ready = df_final[cols_to_keep].copy()

# 4. TRATAMIENTO DE CATEGORÍAS (Para evitar el error de dtypes)
cat_features = [
    'is_sponsored', 'buy_box_availability', 'product_category', 
    'brand_cat', 'spec_resolution_cat', 'is_best_seller', 'product_segment'
]

for col in cat_features:
    if col in new_product_ready.columns:
        new_product_ready[col] = new_product_ready[col].astype(str).astype('category')

print("✅ Producto con Embeddings listo para predicción.")
print(f"Dimensiones: {new_product_ready.shape}") # Debería ser (1, 71) aprox. (21 base + 50 emb)

d:\Documentos\diego\Master\Amazon-Smart-Pricing\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Producto con Embeddings listo para predicción.
Dimensiones: (1, 71)


### Escoger modelo

In [69]:
# 1. Cargar Modelo y Metadatos
# ---------------------------------------------------------
nombre_modelo = 'xgb_wjunwei_COMPONENTES.json' 
path_modelo = os.path.join('Modelos_Especialistas', nombre_modelo)
path_meta = os.path.join('Modelos_Especialistas', 'meta_wjunwei_COMPONENTES.pkl')

# Cargar modelo nativo
best_model = xgb.XGBRegressor()
best_model.load_model(path_modelo)

# Cargar el "ADN" de los datos
metadata = joblib.load(path_meta)
saved_dtypes = metadata['dtypes']
saved_categories = metadata['categories']

print(f"✅ Modelo y metadatos cargados.")

# 2. Alineación de Columnas
# ---------------------------------------------------------
# Usamos las columnas guardadas en los metadatos para asegurar el orden perfecto
model_features = metadata['columns']
df_to_predict = new_product.reindex(columns=model_features).reset_index(drop=True)

# 3. Réplica de Tipos usando los Metadatos
# ---------------------------------------------------------
for col in model_features:
    target_dtype = saved_dtypes[col]
    
    # Verificamos si era categoría en el entrenamiento
    if isinstance(target_dtype, pd.CategoricalDtype):
        # 1. Convertir a string para limpiar
        df_to_predict[col] = df_to_predict[col].astype(str).astype('category')
        
        # 2. INYECTAR LAS CATEGORÍAS GUARDADAS (La clave del éxito)
        # Esto le dice a Pandas: "Usa exactamente estas etiquetas en este orden"
        if col in saved_categories:
            df_to_predict[col] = df_to_predict[col].cat.set_categories(saved_categories[col])
            
    else:
        # Numéricos
        df_to_predict[col] = pd.to_numeric(df_to_predict[col], errors='coerce').astype(target_dtype)

# 4. Predicción
# ---------------------------------------------------------
try:
    pred_log = best_model.predict(df_to_predict)
    precio_final = np.expm1(pred_log[0])
    
    print(f"🎯 Predicción final: {precio_final:.2f} €")
    
except Exception as e:
    print(f"❌ Error: {e}")

✅ Modelo y metadatos cargados.
🎯 Predicción final: 35.36 €
